In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
 
URL = ("https://raw.githubusercontent.com/justmarkham/"
       "pycon-2016-tutorial/master/data/sms.tsv")
sms = pd.read_csv(URL, sep="\t", header=None, names=["label", "message"])
X_sms, y_sms = sms["message"], (sms["label"] == "spam").astype(int)
Xtr, Xte, ytr, yte = train_test_split(X_sms, y_sms, test_size=0.25,
                                      stratify=y_sms, random_state=42)
pipe = Pipeline([("tfidf", TfidfVectorizer()),
                 ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr, ytr)
proba = pipe.predict_proba(Xte)[:, 1]
proba1 = pipe.predict_proba(Xte)

print(proba1) # xác suất ham - xác suất spam

[[0.94661024 0.05338976]
 [0.45227354 0.54772646]
 [0.9631982  0.0368018 ]
 ...
 [0.25303827 0.74696173]
 [0.9270492  0.0729508 ]
 [0.92131043 0.07868957]]


In [7]:
from sklearn.metrics import classification_report

y_true = ["A"] * 8 + ["B"] * 2
y_hat = ["A"] * 8 + ["A","B"]
print(classification_report(y_true, y_hat,digits=3))
print("macro: ", round(f1_score(y_true,y_hat, average="macro"),3))
print("weighted: ", round(f1_score(y_true,y_hat,average="weighted"),3))



              precision    recall  f1-score   support

           A      0.889     1.000     0.941         8
           B      1.000     0.500     0.667         2

    accuracy                          0.900        10
   macro avg      0.944     0.750     0.804        10
weighted avg      0.911     0.900     0.886        10

macro:  0.804
weighted:  0.886


In [ ]:
#  Baseline M3 ở cấp file
from pathlib import Path
from sklearn.dummy import DummyClassifier

df = pd.read_csv(Path("..") / "data" / "raw" / "evtx_data.csv", low_memory=False)
files = df.groupby("EVTX_FileName")["EVTX_Tactic"].first().reset_index(name="tactic")
print(files)
print(len(files), "file")
print(files["tactic"].value_counts().to_dict())

Xf, yf =files["EVTX_FileName"], files["tactic"]
pred = DummyClassifier(strategy="most_frequent").fit(Xf,yf).predict(Xf)
print("đoán toàn:", pred[0])
print("macro   :", round(f1_score(yf, pred, average="macro", zero_division=0), 3)) #  # gán 0, không cảnh báo(nếu preision = 0 (tp=0 vì đoán toàn hàm))
print("weighted:", round(f1_score(yf, pred, average="weighted", zero_division=0), 3))

                                         EVTX_FileName                tactic
0    4624 LT3 AnonymousLogon Localhost - JuicyPotat...  Privilege Escalation
1                       4765_sidhistory_add_t1178.evtx  Privilege Escalation
2                 4794_DSRM_password_change_t1098.evtx     Credential Access
3      ACL_ForcePwd_SPNAdd_User_Computer_Accounts.evtx     Credential Access
4        CA_4624_4625_LogonType2_LogonProc_chrome.evtx     Credential Access
..                                                 ...                   ...
244          tunna_iis_rdp_smb_tunneling_sysmon_3.evtx   Command and Control
245           win10_4703_SeDebugPrivilege_enabled.evtx  Privilege Escalation
246  windows_bits_4_59_60_lolbas desktopimgdownldr....             Execution
247                    wmi_remote_registry_sysmon.evtx      Lateral Movement
248                       wmighost_sysmon_20_21_1.evtx           Persistence

[249 rows x 2 columns]
249 file
{'Privilege Escalation': 61, 'Lateral Movem

In [ ]:
# class_weight="balanced"
pipe_b = Pipeline([("tfidf",TfidfVectorizer()),
                   ("clf",LogisticRegression(max_iter=1000,class_weight="balanced"))])
y_b = pipe_b.fit(Xtr,ytr).predict(Xte)
y_0 = pipe.predict(Xte)
for name, yp in [("không cân bằng", y_0), ("balanced", y_b)]:
    print(f"{name:15s} P={precision_score(yte, yp):.3f} R={recall_score(yte, yp):.3f}"
          f" F1={f1_score(yte, yp):.3f}")
n0, n1 = np.bincount(ytr) # Đếm số lần xuất hiện của 0, 1, 2… trong mảng số nguyên
print("trọng số lớp:", round(len(ytr) / (2 * n0), 3), round(len(ytr) / (2 * n1), 3))
# trọng số của lớp c = tổng số mẫu / (số lớp × số mẫu của lớp c)

không cân bằng  P=1.000 R=0.791 F1=0.884
balanced        P=0.946 R=0.936 F1=0.941
trọng số lớp: 0.577 3.731


In [ ]:
#  PR-AUC và mốc của nó
from sklearn.metrics import average_precision_score
ap = average_precision_score(yte,proba) # PR-AUC trên MỌI ngưỡng Diện tích dưới đường precision–recall khi quét mọi ngưỡng.
print("PR-AUC:", round(ap, 3), "| mốc đoán mò:", round(yte.mean(), 3))
proba_b = pipe_b.predict_proba(Xte)[:,1]
print("PR-AUC balanced:", round(average_precision_score(yte, proba_b), 3))
rand = np.random.default_rng(42).random(len(yte))   # "mô hình" tung xúc xắc
print("PR-AUC ngẫu nhiên:", round(average_precision_score(yte, rand), 3))


PR-AUC: 0.974 | mốc đoán mò: 0.134
PR-AUC balanced: 0.975
